# Advanced 1 — Текст-ембединги підписів (MiniLM)

ТЗ явно дозволяє **локальні текст-ембединги**. Кодуємо підписи моделлю `all-MiniLM-L6-v2` (384-вимірні вектори), стискаємо SVD до 24 вимірів (fit **лише на train**), додаємо до базових фічей і чесно перевіряємо, **чи дають вони сигнал**.

> Fallback: якщо `sentence-transformers` недоступний — TF-IDF + TruncatedSVD (теж локально). Залежності: `%run 03_data_prep.ipynb` (дає `b`, `features`, `config`).

In [ ]:
%run 03_data_prep.ipynb

### Ембедер: MiniLM (основний) з TF-IDF-fallback

In [ ]:
import numpy as np, warnings; warnings.filterwarnings('ignore')
def make_text_matrix(train_texts, test_texts, n_comp=24, seed=42):
    from sklearn.decomposition import TruncatedSVD
    train_texts = [str(t) for t in train_texts]; test_texts = [str(t) for t in test_texts]
    try:
        from sentence_transformers import SentenceTransformer
        enc = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
        Etr = np.asarray(enc.encode(train_texts, show_progress_bar=False))
        Ete = np.asarray(enc.encode(test_texts, show_progress_bar=False))
        name = 'MiniLM-384d'
    except Exception as e:
        from sklearn.feature_extraction.text import TfidfVectorizer
        vec = TfidfVectorizer(max_features=2000, ngram_range=(1,2)).fit(train_texts)  # fit лише train
        Etr = vec.transform(train_texts); Ete = vec.transform(test_texts); name = 'TF-IDF'
    svd = TruncatedSVD(n_components=n_comp, random_state=seed).fit(Etr)  # SVD fit лише на train
    cols = [f'emb_{i}' for i in range(n_comp)]
    import pandas as pd
    return (pd.DataFrame(svd.transform(Etr), columns=cols),
            pd.DataFrame(svd.transform(Ete), columns=cols), name)

Etr, Ete, emb_name = make_text_matrix(b['train_df']['description'], b['test_df']['description'])
print('ембединги:', emb_name, '| shape train:', Etr.shape)

### Порівняння: базові фічі vs базові + ембединги

In [ ]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.metrics import roc_auc_score, brier_score_loss

Xtr, ytr, Xte, yte = b['X_train'], b['y_train'], b['X_test'], b['y_test']
Xtr_emb = pd.concat([Xtr.reset_index(drop=True), Etr.reset_index(drop=True)], axis=1)
Xte_emb = pd.concat([Xte.reset_index(drop=True), Ete.reset_index(drop=True)], axis=1)

def lr(): return Pipeline([('imp',SimpleImputer(strategy='median')),('sc',StandardScaler()),
                           ('clf',LogisticRegression(max_iter=2000,C=0.5,random_state=42))])
cv = StratifiedKFold(5, shuffle=True, random_state=42)
def report(name, Xa, Xb):
    m = lr().fit(Xa, ytr); p = m.predict_proba(Xb)[:,1]
    oof = cross_val_predict(lr(), Xa, ytr, cv=cv, method='predict_proba')[:,1]
    print(f'{name:16s} test AUC={roc_auc_score(yte,p):.3f}  test Brier={brier_score_loss(yte,p):.3f}'
          f'  OOF AUC={roc_auc_score(ytr,oof):.3f}')
print('LogReg, temporal test (n=%d):' % len(yte))
report('base (32)', Xtr, Xte)
report('base+emb (%d)' % Xtr_emb.shape[1], Xtr_emb, Xte_emb)

### Висновок

Чесно інтерпретуй дельту: на цьому крихітному дрейфованому датасеті ембединги підписів зазвичай дають **малу або нульову** перевагу (вірусність визначає відео/звук/тренд, не текст). Дивись на **OOF AUC** (без дрейфу) — якщо там лифт є, сигнал реальний, але слабкий. Якщо нема — це теж результат: текст-ембединги тут не виправдані, і простий набір фічей лишається кращим (бритва Оккама).